In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import pickle

In [ ]:
################ Load model results
# Demand model results
with open('demand_model_results.pkl', 'rb') as f:
     demand_param = pickle.load(f)

# Charging station model results
with open('charging_station_model_results.pkl', 'rb') as f:
     charging_param = pickle.load(f)


In [ ]:
################ Calculate status quo


In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# ======================
# 1. Data Preparation (Mock Data)
# ======================
# Assume we have the following variables (similar to the original paper):
# - `X1`: Product characteristics (e.g., horsepower, weight)
# - `price`: Vehicle price (with/without subsidies)
# - `EV`: Dummy for electric vehicles
# - `household`: Market size (number of households)
# - `N`: Number of charging stations (endogenous)
# - `EVSE_subsidy`: Charging station subsidies

n_products = 100  # Number of vehicle models
n_markets = 19    # Number of counties

# Generate mock data
data = pd.DataFrame({
    'product_id': np.arange(n_products),
    'market_id': np.random.choice(n_markets, n_products),
    'EV': np.random.binomial(1, 0.2, n_products),  # 20% EVs
    'price': np.random.uniform(100, 500, n_products),
    'X1': np.random.normal(0, 1, n_products),  # Product characteristic
    'household': np.random.randint(1000, 5000, n_products),  # Market size
    'N': np.random.poisson(50, n_products),  # Charging stations
    'EVSE_subsidy': np.random.uniform(0, 10, n_products)  # Station subsidy
})

# Add unobserved product characteristics (xi)
data['xi'] = np.random.normal(0, 0.5, n_products)

# ======================
# 2. Nested Logit Demand Model
# ======================
# Parameters (fixed, estimated elsewhere)
alpha = -0.5    # Price coefficient
beta_X1 = 0.3   # Coefficient for X1
beta_N = 0.4    # Network effect (log stations)
lambda_ev = 0.7   # Nesting parameter (EVs)
lambda_nonEV = 0.9 # Nesting parameter (non-EVs)

def nested_logit_shares(V, EV, lambda_ev, lambda_nonEV):
    """Compute market shares using nested logit."""
    # Split utilities by nest (EVs vs. non-EVs)
    V_ev = V[EV == 1]
    V_nonEV = V[EV == 0]
    
    # Logsum terms for each nest
    if len(V_ev) > 0:
        logsum_ev = np.log(np.sum(np.exp(V_ev / lambda_ev)))
    else:
        logsum_ev = -np.inf  # No EVs in this market
    
    if len(V_nonEV) > 0:
        logsum_nonEV = np.log(np.sum(np.exp(V_nonEV / lambda_nonEV)))
    else:
        logsum_nonEV = -np.inf  # No non-EVs
    
    # Denominator (total nest attractiveness)
    D = np.exp(logsum_ev)**lambda_ev + np.exp(logsum_nonEV)**lambda_nonEV
    
    # Compute shares
    shares = np.zeros_like(V)
    shares[EV == 1] = np.exp(V_ev / lambda_ev) * np.exp((lambda_ev - 1) * logsum_ev) / D
    shares[EV == 0] = np.exp(V_nonEV / lambda_nonEV) * np.exp((lambda_nonEV - 1) * logsum_nonEV) / D
    
    # Include outside good (s0 = 1 - sum(sj))
    shares = shares / (1 + np.sum(shares))
    return shares

# ======================
# 3. Counterfactual Setup
# ======================
def compute_equilibrium(data, price_subsidy=0, station_subsidy=0):
    """Compute equilibrium shares and stations given subsidies."""
    # Apply subsidies
    data['price_adj'] = data['price'] - price_subsidy * data['EV']  # EV price subsidy
    data['N_adj'] = data['N'] + station_subsidy * data['EVSE_subsidy']  # Station subsidy effect
    
    # Compute utilities
    V = beta_X1 * data['X1'] + alpha * data['price_adj'] + beta_N * np.log(data['N_adj'] + 1) + data['xi']
    
    # Compute market shares
    data['shares'] = nested_logit_shares(V, data['EV'], lambda_ev, lambda_nonEV)
    
    # Compute EV sales
    ev_sales = np.sum(data['shares'] * data['household'] * data['EV'])
    
    # Compute government spending (simplified)
    gov_spending = (
        np.sum(price_subsidy * data['shares'] * data['household'] * data['EV']) +  # EV subsidies
        np.sum(station_subsidy * data['EVSE_subsidy'])  # Station subsidies
    )
    
    return ev_sales, gov_spending

# ======================
# 4. Run Counterfactuals
# ======================
# Baseline (no subsidies)
ev_baseline, spending_baseline = compute_equilibrium(data, 0, 0)

# Scenario 1: Only EV purchase subsidy (e.g., 10k NOK per EV)
ev_subsidy_only, spending_subsidy = compute_equilibrium(data, price_subsidy=10, station_subsidy=0)

# Scenario 2: Only station subsidy (e.g., 10k NOK per station)
station_subsidy_only, spending_station = compute_equilibrium(data, price_subsidy=0, station_subsidy=10)

# Scenario 3: Both subsidies
both_subsidies, spending_both = compute_equilibrium(data, price_subsidy=10, station_subsidy=10)

# ======================
# 5. Results Table
# ======================
results = pd.DataFrame({
    'Policy': ['No subsidies', 'EV subsidy only', 'Station subsidy only', 'Both subsidies'],
    'ΔEV Purchases': [
        0,
        ev_subsidy_only - ev_baseline,
        station_subsidy_only - ev_baseline,
        both_subsidies - ev_baseline
    ],
    'ΔGov Spending (M NOK)': [
        0,
        (spending_subsidy - spending_baseline) / 1e6,
        (spending_station - spending_baseline) / 1e6,
        (spending_both - spending_baseline) / 1e6
    ],
    'Cost-Effectiveness (EVs/M NOK)': [
        0,
        (ev_subsidy_only - ev_baseline) / ((spending_subsidy - spending_baseline) / 1e6),
        (station_subsidy_only - ev_baseline) / ((spending_station - spending_baseline) / 1e6),
        (both_subsidies - ev_baseline) / ((spending_both - spending_baseline) / 1e6)
    ]
})

print(results)